In [1]:
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback
from datasets import load_dataset
import logging
import os

os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"


# Configure logging
logging.basicConfig(format='%(asctime)s - %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# ---------------------------
# 1. Model Initialization (from scratch)
# ---------------------------
# We use the configuration of a popular small-scale LLM (facebook/opt-125m)
# but initialize the model randomly (i.e. train from scratch)
model_name = "facebook/opt-125m"
model_name = "EleutherAI/gpt-neo-125M"
config = AutoConfig.from_pretrained(model_name)  # load config; do NOT load pretrained weights
model = AutoModelForCausalLM.from_config(config)   # randomly initialized model

# Load tokenizer (we can reuse the pretrained tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # assign PAD token if missing

# Enable gradient checkpointing to reduce memory usage (at the cost of additional compute)
model.gradient_checkpointing_enable()
logger.info(f"Initialized model from scratch with configuration from '{model_name}'.")

# ---------------------------
# 2. Dataset Preparation
# ---------------------------
# Load the Wikitext-2 dataset as our general-purpose text corpus.
# For a quick profiling run, we use only a small subset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train_dataset = dataset["train"].select(range(1000))  # limit to 1000 examples for this demo

# Tokenization: convert text to token IDs (truncated to a maximum length)
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator: handles padding and prepares labels for causal LM (labels equal to input_ids)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ---------------------------
# 3. Trainer Setup with Memory Profiling Callback
# ---------------------------
# TrainingArguments are set to run only 3 steps and use a small batch size suitable for an 8–12GB GPU.
training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=2,
    max_steps=3,  # run only 3 training iterations for profiling
    gradient_accumulation_steps=1,
    fp16=False,  # using full precision; set True if your GPU supports mixed precision to save memory
    logging_steps=1,
    report_to=[],  # disable external logging (e.g., wandb)
    disable_tqdm=True
)

# Custom callback to log GPU memory usage at the start and end of each step
class MemoryProfilerCallback(TrainerCallback):
    def on_step_begin(self, args, state, control, **kwargs):
        torch.cuda.reset_peak_memory_stats()  # reset peak memory stats at beginning of each step

    def on_step_end(self, args, state, control, **kwargs):
        alloc = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        peak = torch.cuda.max_memory_allocated()
        step = state.global_step
        logger.info(f"Step {step}: allocated={alloc/1024**2:.2f} MB, reserved={reserved/1024**2:.2f} MB, peak={peak/1024**2:.2f} MB")
        return control

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[MemoryProfilerCallback()]
)

# ---------------------------
# 4. Training with PyTorch Profiler
# ---------------------------
# We set up the PyTorch Profiler to capture CPU and CUDA activities with memory profiling enabled.
activities = [torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA]
profiler = torch.profiler.profile(
    activities=activities,
    profile_memory=True,  # capture tensor memory allocations
    record_shapes=True    # record shapes of tensors for deeper insight
)

profiler.start()
trainer.train()  # run 3 training iterations
profiler.stop()

# Print a summary of the profiler's memory usage by CUDA operation (top 10 ops)
print("Profiler Memory Usage Summary (top CUDA ops):")
print(profiler.key_averages().table(sort_by="self_cuda_memory_usage", row_limit=10))


/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-03 23:48:29,786 - INFO - Initialized model from scratch with configuration from 'EleutherAI/gpt-neo-125M'.
Map: 100%|██████████| 1000/1000 [00:00<00:00, 41552.45 examples/s]
/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/torch/nn/parallel/data_parallel.py:37: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 1 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/home/glaswigian/miniconda3/envs/

{'loss': 5.3601, 'grad_norm': 6.9823079109191895, 'learning_rate': 3.3333333333333335e-05, 'epoch': 0.004}


2025-03-03 23:48:41,193 - INFO - Step 2: allocated=1505.72 MB, reserved=3334.00 MB, peak=3138.66 MB


{'loss': 5.1552, 'grad_norm': 5.0893120765686035, 'learning_rate': 1.6666666666666667e-05, 'epoch': 0.008}


2025-03-03 23:48:43,450 - INFO - Step 3: allocated=1505.71 MB, reserved=3334.00 MB, peak=3129.93 MB


{'loss': 5.1045, 'grad_norm': 5.98524284362793, 'learning_rate': 0.0, 'epoch': 0.012}
{'train_runtime': 10.1208, 'train_samples_per_second': 1.186, 'train_steps_per_second': 0.296, 'train_loss': 5.20662260055542, 'epoch': 0.012}
Profiler Memory Usage Summary (top CUDA ops):
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ----